<a href="https://colab.research.google.com/github/PeroronShine/education_fefu_2/blob/main/Park_math/%D0%B2%D1%8B%D1%87%D0%BC%D0%B5%D1%827.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

def generate_integer_well_conditioned_matrix(n, cond_threshold=500, max_attempts=1000):

    for attempt in range(max_attempts):
        A_off = np.random.randint(-10, 11, size=(n, n))
        np.fill_diagonal(A_off, 0)

        row_sums = np.sum(np.abs(A_off), axis=1)
        diag_vals = row_sums + np.random.randint(1, 6, size=n)

        A = A_off.copy()
        np.fill_diagonal(A, diag_vals)

        # Проверяем число обусловленности
        cond_A = np.linalg.cond(A.astype(float))
        if cond_A < cond_threshold and not np.isnan(cond_A):
            det_A = np.linalg.det(A.astype(float))
            if abs(det_A) > 1e-8:
                print(f"Целочисленная матрица сгенерирована (попытка {attempt + 1}), cond(A) = {cond_A:.2f}")
                return A.astype(int)

    raise RuntimeError(f"Не удалось сгенерировать подходящую целочисленную матрицу за {max_attempts} попыток.")

def simple_iteration_method(A, b, eps=1e-6, max_iter=10000, verbose=True):
    n = A.shape[0]
    A = A.astype(float)
    b = b.astype(float)

    D = np.diag(np.diag(A))
    if np.any(np.diag(D) == 0):
        raise ValueError("Нулевые диагональные элементы — метод не применим.")

    D_inv = np.diag(1.0 / np.diag(D))
    B = np.eye(n) - D_inv @ A
    g = D_inv @ b

    # проверка сходимости
    eigvals_B = np.linalg.eigvals(B)
    spectral_radius = np.max(np.abs(eigvals_B))

    #смотрим максимальный модуль собственных значений матрицы B
    if spectral_radius >= 1:
        print(f"B = {spectral_radius} ≥ 1")

    x_prev = np.zeros(n)
    if verbose:
        print(f"\n{'Итерация':<8} {'||Δx||_∞':<15} {'x_k'}")
        print("-" * 70)

    for k in range(1, max_iter + 1):
        x_next = B @ x_prev + g

        diff = np.linalg.norm(x_next - x_prev, ord=np.inf)

        if verbose:
            x_str = np.array2string(x_next, formatter={'float_kind': lambda x: f"{x}"})
            print(f"{k:<8} {diff:<15.2e} {x_str}")
        if diff < eps:
            if verbose:
                print(f"\nСходимость достигнута на итерации {k}.")
            return x_next, k
        x_prev = x_next

    raise RuntimeError(f"Не сошлось за {max_iter} итераций.")


In [20]:
np.set_printoptions(precision=6, suppress=True, linewidth=120)

n = 10
cond_threshold = 500

A = generate_integer_well_conditioned_matrix(n, cond_threshold=cond_threshold)
x_true = np.random.randn(n)

Целочисленная матрица сгенерирована (попытка 1), cond(A) = 2.43


In [21]:
eps = 1e-10

#x_true = [2, 6, 1, 1]
b = A @ x_true

print("Целочисленная матрица A:")
print(A)
print("\nИзвестное решение x_true:")
print(x_true)
print("\nВектор b:")
print(b)

x_approx, n_iter = simple_iteration_method(A, b, eps=eps, verbose=True)

residual = np.linalg.norm(A @ x_approx - b, ord=np.inf)
print(f"\nПриближённое решение x ≈ {x_approx}")
print(f"Невязка ||Ax^(k) - b||_∞ = {residual:.2e}")
print(f"Количество итераций: {n_iter}")

Целочисленная матрица A:
[[ 60   4  -1  10   5  10   7  -1 -10  -7]
 [ -6  52   7  -3   6  10  -7  -1   2  -9]
 [ -6   4  50 -10   1  -2  10   6  -5   4]
 [  1  -8 -10  45  -4  -5   5  -2   2   7]
 [  8   2   1 -10  57   6  -9  10   1   9]
 [ -9   3   1  -2  -8  44  -9   1   4   5]
 [ -3  -5   8  -3   3   8  47  -5   3  -5]
 [ 10  10   2  -1  -9  -6   2  45   0  -2]
 [ -7   0   8  -1  -1   4  -7  -4  37  -4]
 [  1   0   8  -4  -8  -4  -5   3   8  43]]

Известное решение x_true:
[ 0.870235  0.18642   0.025285  0.866179 -0.495235 -1.440313 -1.902409  0.639023 -0.093889  1.21479 ]

Вектор b:
[  23.196453  -13.766693  -19.349228   44.812245   -3.82092   -44.932741 -117.911247   45.371553   -9.593848
   70.244885]

Итерация ||Δx||_∞        x_k
----------------------------------------------------------------------
1        2.51e+00        [0.3866075528807774 -0.2647440947447542 -0.3869845614759316 0.9958276755008331 -0.06703369275549367 -1.0211986529729207
 -2.508749941241495 1.0082567442861